# 超市销售与库存分析（工程化）

### 导入模块

In [1]:
import sys
from pathlib import Path

# 将 src 目录加入 Python 路径（如果尚未加入）
src_path = Path.cwd() / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 导入自定义模块
from config import OUTPUT_DIR
from data_loader import load_all_data
from preprocessing import preprocess_all
from metrics import (
    daily_net_sales,
    product_return_rate,
    inventory_turnover,
    promotion_effect,
    abc_classification
)
from inventory_advanced import check_return_date_anomaly, interpolate_inventory
from report_exporter import export_analysis

### 加载原始数据并展示

In [2]:
# 1. 加载数据
sales, returns, inventory = load_all_data()

print("sales 前5行:")
print(sales.head())
print("\nreturns 前5行:")
print(returns.head())
print("\ninventory 前5行:")
print(inventory.head())

print("\nsales info:")
print(sales.info())
print("\nreturns info:")
print(returns.info())
print("\ninventory info:")
print(inventory.info())

sales 前5行:
   sale_id store_id  product_id  sale_date  quantity  unit_price  discount
0        1        C         120 2022-12-15         8      121.74      0.00
1        2        C         111 2022-03-29         5      122.22      0.05
2        3        B         102 2022-12-10         6        5.15      0.30
3        4        A         112 2022-11-10         6        6.38      0.00
4        5        C         110 2022-07-07         3       79.58      0.30

returns 前5行:
   return_id  sale_id  product_id return_date  return_quantity  refund_amount
0          1      NaN         108  2022-10-08              1.0            NaN
1          2      NaN         107  2022-03-11              3.0            NaN
2          3    409.0         119  2022-09-28              1.0          41.41
3          4   3119.0         116  2022-12-31              7.0         660.28
4          5   4935.0         104  2022-09-23              5.0         800.00

inventory 前5行:
  store_id  product_id       date  stock_

### 预处理

In [3]:
# 2. 预处理（缺失值填充、异常值删除、计算列）
sales_clean, returns_clean, inventory_clean = preprocess_all(sales, returns, inventory)

print("预处理完成。")
print(f"sales 清洗后形状: {sales_clean.shape}")
print(f"returns 清洗后形状: {returns_clean.shape}")
print(f"inventory 清洗后形状: {inventory_clean.shape}")

预处理完成。
sales 清洗后形状: (4949, 8)
returns 清洗后形状: (990, 7)
inventory 清洗后形状: (17488, 4)


### 每日净销售额

In [4]:
# 3. 各门店每日净销售额
daily_sales = daily_net_sales(sales_clean)

print("每日净销售额（前5行）:")
print(daily_sales.head())

每日净销售额（前5行）:
  store_id  sale_date  daily_net_sales
0        A 2022-01-01        2777.4105
1        A 2022-01-02         585.4775
2        A 2022-01-03         864.2400
3        A 2022-01-04        3701.1060
4        A 2022-01-05        1429.3890


### 商品退货率 Top5

In [5]:
# 4. 商品退货率 Top5
return_rate_top5 = product_return_rate(sales_clean, returns_clean)

print("商品退货率 Top5:")
print(return_rate_top5)

商品退货率 Top5:
    product_id  total_sales_qty  total_return_qty return_rate
0          101             1093             211.0       19.3%
16         117             1238             184.0      14.86%
2          103             1116             164.0       14.7%
10         111             1008             147.0      14.58%
9          110             1360             193.0      14.19%


### 库存周转最慢 Top10

In [6]:
# 5. 库存周转最慢 Top10
slow_turnover = inventory_turnover(sales_clean, inventory_clean)

print("库存周转最慢 Top10:")
print(slow_turnover.head(10))

库存周转最慢 Top10:
   store_id  product_id         平均库存  全年销量      周转次数         周转天数
10        A         111  1697.583333   278  0.163762  2228.841427
38        B         119  1637.166667   372  0.227222  1606.359767
25        B         106  1528.833333   379  0.247901  1472.359279
0         A         101  1566.166667   390  0.249016  1465.771368
17        A         118  1480.666667   375  0.253264  1441.182222
29        B         110  1478.666667   384  0.259693  1405.503472
55        C         116  1493.583333   388  0.259778  1405.046177
13        A         114  1377.500000   361  0.262069  1392.763158
50        C         111  1159.000000   304  0.262295  1391.562500
11        A         112  1416.333333   373  0.263356  1385.956211


### 促销效果评估

In [7]:
# 6. 促销效果评估
promo_effect = promotion_effect(sales_clean)

print("促销效果评估:")
print(promo_effect)

促销效果评估:
  store_id  net_sales促销日  net_sales非促销日 是否有效
0        A    743.753230    1702.756895   无效
1        B    839.767783    1759.816686   无效
2        C    820.038002    1638.318287   无效


### 商品 ABC 分类

In [8]:
# 7. ABC 分类
abc_result = abc_classification(sales_clean)

print("ABC 分类结果（前10行）:")
print(abc_result.head(10))

ABC 分类结果（前10行）:
    product_id          销售额   类别
3          104  144031.2900  A 类
15         116  137415.6190  A 类
9          110  135531.3905  A 类
7          108  135097.6500  A 类
12         113  132556.6680  A 类
8          109  130449.3760  A 类
6          107  130118.7005  A 类
14         115  126521.4330  A 类
5          106  123147.1645  A 类
11         112  122410.0395  A 类


### 进阶分析：退货日期异常检查

In [9]:
# 8. 检查退货日期早于销售日期的异常记录数
anomaly_cnt = check_return_date_anomaly(sales_clean, returns_clean)

print(f"退货日期早于销售日期的异常记录数: {anomaly_cnt}")

退货日期早于销售日期的异常记录数: 0


### 进阶分析：库存线性插值补全

In [10]:
# 9. 库存线性插值补全（展示前10行验证）
inventory_full = interpolate_inventory(inventory_clean)

print("补全后的库存表（前10行）:")
print(inventory_full.head(10))

补全后的库存表（前10行）:
                                stock_on_hand
store_id product_id                          
A        101        2022-01-01          479.0
                    2022-01-02          460.0
                    2022-01-03          453.0
                    2022-01-04          461.0
                    2022-01-05          441.0
                    2022-01-06          454.0
                    2022-01-07          452.0
                    2022-01-08          448.0
                    2022-01-09          444.0
                    2022-01-10          469.0


### 导出 Excel 报告

In [11]:
# 10. 导出 Excel 报告（含格式）
excel_path = export_analysis(
    daily_net_sales_df=daily_sales,
    top_return_df=return_rate_top5,
    top_slow_turnover_df=slow_turnover,
    promotion_effect_df=promo_effect,
    abc_class_df=abc_result
)
print(f"Excel 报告已保存至: {excel_path}")

Excel 报告已保存至: C:\Users\IJP25XG\中文\outputs\supermarket_analysis_20260418.xlsx
